In [ ]:
# ============================================================
# Cell 1: ENVIRONMENT GATE — Colab A100 ONLY
# ============================================================
# MANDATORY: This cell must pass before ANY training.
# Refuses local execution. Verifies CUDA GPU.
import os, sys

_cwd = os.getcwd()
_is_local = _cwd.startswith("/Users/") or (_cwd.startswith("/home/") and "content" not in _cwd)

import torch
_has_cuda = torch.cuda.is_available()

if _is_local or not _has_cuda:
    print("=" * 65)
    print("  BLOCKED: This notebook must run on Google Colab with CUDA GPU")
    print(f"  Current dir : {_cwd}")
    print(f"  CUDA        : {_has_cuda}")
    print("=" * 65)
    print("\n  Runtime → Change runtime type → A100 GPU")
    raise SystemExit("Refusing local execution. Use Colab A100.")

# Passed — CUDA environment verified
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)

print(f"{'='*65}")
print(f"  ENVIRONMENT VERIFIED")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram_gb:.1f} GB")
print(f"  Compute cap  : {cc[0]}.{cc[1]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")
print(f"  Working dir  : {_cwd}")
print(f"{'='*65}")
os.system("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

  ENVIRONMENT VERIFIED
  GPU          : NVIDIA A100-SXM4-40GB
  VRAM         : 42.4 GB
  Compute cap  : 8.0
  PyTorch      : 2.9.0+cu126
  CUDA         : 12.6
  Working dir  : /content


0

In [ ]:
# ============================================================
# Cell 2: Clone Repo + Install Dependencies
# ============================================================
import subprocess, os, sys, pathlib

REPO_URL = "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git"
PROJ_ROOT = "/content/nst"

if not os.path.isdir(os.path.join(PROJ_ROOT, "data")):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, PROJ_ROOT], check=True)
else:
    print("Repository already cloned. Pulling latest...")
    subprocess.run(["git", "-C", PROJ_ROOT, "pull", "--ff-only"], check=True)

os.chdir(PROJ_ROOT)
if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)

# Install package + deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJ_ROOT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets==2.21.0", "peft>=0.9", "transformers>=4.40",
                "accelerate", "sentencepiece", "protobuf"], check=True)

print(f"\nProject root: {PROJ_ROOT}")
print(f"Python: {sys.executable}")

# GPU auto-config
import torch
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
cc = (props.major, props.minor)
supports_bf16 = cc >= (8, 0)

if supports_bf16:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# Determine batch config based on VRAM
if vram_gb >= 35:
    BS, GA = 32, 2
elif vram_gb >= 20:
    BS, GA = 24, 2
else:
    BS, GA = 16, 2

GPU_OVERRIDES = {
    "train": {
        "batch_size": BS,
        "grad_accum_steps": GA,
        "bf16": supports_bf16,
        "fp16": not supports_bf16,
        "tf32": supports_bf16,
        "fused_optimizer": True,
        "num_workers": 4,
    }
}

DEVICE = "cuda"
prec = "BF16" if supports_bf16 else "FP16"
print(f"\nGPU config: bs={BS}×{GA}={BS*GA} effective, {prec}, VRAM={vram_gb:.0f}GB")

Repository already cloned. Pulling latest...

Project root: /content/nst
Python: /usr/bin/python3

GPU config: bs=32×2=64 effective, BF16, VRAM=42GB


In [ ]:
# ============================================================
# Cell 3: Verify datasets version compatibility
# ============================================================
import datasets
print(f"datasets version: {datasets.__version__}")
_ds_major = int(datasets.__version__.split(".")[0])
assert _ds_major < 3, (
    f"datasets {datasets.__version__} doesn't support FEVER loading scripts. "
    "Run: pip install datasets==2.21.0 then restart the kernel."
)
print("OK — compatible with FEVER loading script")

datasets version: 2.21.0
OK — compatible with FEVER loading script


In [ ]:
# ============================================================
# Cell 4: Build Wiki Cache & Verify Evidence Quality
# ============================================================
# The FEVER dataset needs a wiki page cache to resolve evidence
# text from page title + sentence index. Without it, evidence is
# just page titles → NLI accuracy capped at ~60-70%.
import logging, os, time, sys
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Clear stale project modules for fresh imports
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

# ── Step 1: Build wiki cache if missing ──
from data.fever_wiki_cache import build_wiki_cache, cache_stats

cache_path = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
stats = cache_stats(cache_path)
if stats.get("exists"):
    print(f"Wiki cache exists: {stats['n_pages']} pages, {stats['size_mb']:.1f} MB")
else:
    print("Building wiki page cache (one-time, ~5-10 min)...")
    t0 = time.time()
    build_stats = build_wiki_cache(cache_path=cache_path)
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.0f}s: {build_stats['n_found']}/{build_stats['n_needed']} pages")

# ── Step 2: Load small data sample and verify evidence quality ──
from data.fever_dataset import load_fever_splits, print_fever_stats

splits_check = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits_check)

train_items = splits_check["train"]
n_with_evidence = sum(1 for it in train_items if len(it.get("gold_evidence_text", "")) > 30)
pct = 100 * n_with_evidence / max(1, len(train_items))
print(f"\nEvidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) have >30 char evidence")

# Show sample evidence for manual verification
for i in range(min(5, len(train_items))):
    it = train_items[i]
    ev = it.get("gold_evidence_text", "")[:150]
    print(f"\n  [{i}] {it['label']}: {it['claim'][:80]}")
    print(f"       Evidence: {ev}")

if pct > 60:
    print(f"\n{'='*50}")
    print(f"  EVIDENCE CHECK PASSED — {pct:.0f}% coverage")
    print(f"{'='*50}")
elif pct > 20:
    print(f"\n  WARNING: Partial evidence ({pct:.0f}%). Results will be degraded.")
else:
    print(f"\n  CRITICAL: Only {pct:.0f}% have evidence. Wiki cache needed.")
    print(f"  Run: python main.py build-fever-wiki-cache")

fever_wiki_cache | Building FEVER wiki cache...
fever_wiki_cache |   Loading HF fever/v1.0 dataset...


Building wiki page cache (one-time, ~5-10 min)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/311431 [00:00<?, ? examples/s]

Generating labelled_dev split:   0%|          | 0/37566 [00:00<?, ? examples/s]

...

Generating wikipedia_pages split:   0%|          | 0/5416537 [00:00<?, ? examples/s]

fever_wiki_cache |   Pass 2: Streaming 5416537 wiki pages, filtering to 14533 needed titles...
fever_wiki_cache |     Scanned 1894767 pages, found 5000/14533...
fever_wiki_cache |     Scanned 3951899 pages, found 10000/14533...
fever_wiki_cache |   ✅ Wiki cache built: 14363/14533 pages (170 missing) in 451.4s → /content/nst/data/fever_wiki.db (24.2 MB)
fever_wiki_cache |   ⚠️  170 pages not found in wiki_pages split. Evidence for those will use title-only fallback.
fever_wiki_cache |   Manifest written to /content/nst/data/fever_wiki_manifest.json
fever_dataset | Loading FEVER from HuggingFace datasets...


Done in 452s: 14363/14533 pages


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 500 examples (277 with evidence text, 223 without)
fever_dataset |   dev: 200 examples (91 with evidence text, 109 without)
fever_dataset |   Split labelled_dev into dev (180) + dev_test (20)
fever_dataset |   train hash: 99235a28415638b0
fever_dataset |   dev hash: 0b8223f9a96b44c5
fever_dataset |   dev_test hash: 6ffcd5f9dc97f2f4


  FEVER Dataset Statistics

  train: 500 examples
    With gold evidence: 277 (55.4%)
    Label distribution:
      SUPPORTS                274  (54.8%)
      REFUTES                 114  (22.8%)
      NOT ENOUGH INFO         112  (22.4%)
    Split hash: 99235a28415638b0

  dev: 180 examples
    With gold evidence: 83 (46.1%)
    Label distribution:
      SUPPORTS                 65  (36.1%)
      REFUTES                  57  (31.7%)
      NOT ENOUGH INFO          58  (32.2%)
    Split hash: 0b8223f9a96b44c5

  dev_test: 20 examples
    With gold evidence: 8 (40.0%)
    Label distribution:
      SUPPORTS                  5  (25.0%)
      REFUTES                   8  (40.0%)
      NOT ENOUGH INFO           7  (35.0%)
    Split hash: 6ffcd5f9dc97f2f4

Evidence quality: 276/500 (55%) have >30 char evidence

  [0] SUPPORTS: Chitty Chitty Bang Bang is loosely based on another novel.
       Evidence: 

  [1] SUPPORTS: The Republic of Zambia is bordered to the west by Angola.
       Evidence:

In [ ]:
# ============================================================
# Cell 5: Smoke Test — 50 examples (validates pipeline end-to-end)
# ============================================================
import time, gc, json, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("Smoke test: 50 train / 25 dev / 1 epoch (VERI mode)")
print("Purpose: verify pipeline runs end-to-end on CUDA\n")

t0 = time.time()
from training.train_fever_veri import train_fever_veri

results_smoke = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides={
        "data": {"max_train": 50, "max_dev": 25, "dev_sample": 25},
        "train": {"epochs": 1, "eval_every_steps": 25, "patience": 100},
        "io": {"out_dir": "outputs_smoke_veri"},
    }
)
elapsed = time.time() - t0

dev = results_smoke.get("dev", {})
print(f"\nSmoke test complete in {elapsed:.1f}s")
print(f"  Dev acc: {dev.get('accuracy', 'N/A')}")
print(f"  (Accuracy is meaningless at 50 examples — this just validates the pipeline)")

# Verify CUDA was actually used
assert DEVICE == "cuda", "ERROR: Not running on CUDA!"
print(f"\n  Pipeline smoke test PASSED on {torch.cuda.get_device_name(0)}")

train_fever_veri | TF32 enabled
train_fever_veri | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


Smoke test: 50 train / 25 dev / 1 epoch (VERI mode)
Purpose: verify pipeline runs end-to-end on CUDA



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 50 examples (25 with evidence text, 25 without)
fever_dataset |   dev: 25 examples (9 with evidence text, 16 without)
fever_dataset |   Split labelled_dev into dev (22) + dev_test (3)
fever_dataset |   train hash: 5b32717285974de9
fever_dataset |   dev hash: 9056657022ec8b16
fever_dataset |   dev_test hash: d9980171cd29c1e1
train_fever_veri | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 50 examples
    With gold evidence: 25 (50.0%)
    Label distribution:
      SUPPORTS                 24  (48.0%)
      REFUTES                  16  (32.0%)
      NOT ENOUGH INFO          10  (20.0%)
    Split hash: 5b32717285974de9

  dev: 22 examples
    With gold evidence: 7 (31.8%)
    Label distribution:
      SUPPORTS                  8  (36.4%)
      REFUTES                   7  (31.8%)
      NOT ENOUGH INFO           7  (31.8%)
    Split hash: 9056657022ec8b16

  dev_test: 3 examples
    With gold evidence: 2 (66.7%)
    Label distribution:
      SUPPORTS                  2  (66.7%)
      REFUTES                   0  (0.0%)
      NOT ENOUGH INFO           1  (33.3%)
    Split hash: d9980171cd29c1e1

  Evidence quality: 25/50 (50%) have >30 char evidence
    [0] SUPPORTS: Chitty Chitty Bang Bang is loosely based on another novel....
         Evidence: 
    [1] SUPPORTS: The Republic of Zambia is bordered to the west by Angola....
         Evi

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever_veri | Class weights: [0.6944444179534912, 1.0416666269302368, 1.6666666269302368]
train_fever_veri |   backbone: 0.00M params, lr=2e-05
train_fever_veri |   lora: 7.11M params, lr=0.0005
train_fever_veri |   heads: 2.43M params, lr=0.001
train_fever_veri |   adaptive_lambda: 0.00M params, lr=0.003



  Constraint precision pre-check (50 samples):
    NumericalConstraint            precision=0.000 fire_rate=0.060 [NOISY]
    NegationConstraint             precision=1.000 fire_rate=0.060 [OK]
    EntityOverlapConstraint        precision=0.348 fire_rate=0.460 [NOISY]
    EvidenceSufficiencyConstraint  precision=0.400 fire_rate=0.500 [NOISY]
    TemporalConstraint             precision=0.333 fire_rate=0.060 [NOISY]
    HedgeModalityConstraint        precision=0.000 fire_rate=0.000 [NOISY]

  NST-VERI Training: Verification-Enhanced FEVER
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 9.55M / 444.61M
  Adaptive lambda: 4551 params, lambda_max=0.8
  epochs=1, bs=16x2=32
  lr=2e-05, lr_lora=0.0005, lr_heads=0.001
  precision=bf16, compile=False
  Focal loss: no
  Phases: 1→NLI+aux, 2→+contrastive, 3→+constraints
  total_steps=2, warmup=0

  Phase 1 | Epoch 1/1 | β=1.00 γ=0.00 sched=0.00
  Epoch 1/1 (phase 1): loss=1.7585 nli=1.1038 aux=0.6547 con=0.0000 cst=0.0000 | dev_acc

In [ ]:
# ============================================================
# Cell 6: 3K NEURAL BASELINE (Fair Comparison — Same Architecture)
# ============================================================
# DeBERTa-v3-large + LoRA, NO constraints.
# Establishes the ceiling that pure neural achieves on 3k.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k = train_fever_nst(
    "configs/fever_neural_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_neural_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k.json", "w") as f:
    json.dump(results_neural_3k, f, indent=2, default=str)
print(f"\n  Saved to results_neural_3k.json")

train_fever | TF32 enabled for matmul and cuDNN
train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  3K NEURAL BASELINE: DeBERTa-v3-large + LoRA (NO constraints)


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (1586 with evidence text, 1414 without)
fever_dataset |   dev: 1000 examples (428 with evidence text, 572 without)
fever_dataset |   Split labelled_dev into dev (900) + dev_test (100)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: f36abc6968b9527d
fever_dataset |   dev_test hash: ae1018906c1b63d9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 3000 examples
    With gold evidence: 1586 (52.9%)
    Label distribution:
      SUPPORTS               1594  (53.1%)
      REFUTES                 641  (21.4%)
      NOT ENOUGH INFO         765  (25.5%)
    Split hash: 971dd9bf22223f7f

  dev: 900 examples
    With gold evidence: 381 (42.3%)
    Label distribution:
      SUPPORTS                301  (33.4%)
      REFUTES                 276  (30.7%)
      NOT ENOUGH INFO         323  (35.9%)
    Split hash: f36abc6968b9527d

  dev_test: 100 examples
    With gold evidence: 47 (47.0%)
    Label distribution:
      SUPPORTS                 41  (41.0%)
      REFUTES                  27  (27.0%)
      NOT ENOUGH INFO          32  (32.0%)
    Split hash: ae1018906c1b63d9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever | Class weights: [0.6273525953292847, 1.5600624084472656, 1.3071895837783813]
train_fever | Periodic eval uses dev subset: 500/900 examples
train_fever | Params: 7.11M trainable / 442.18M total
train_fever |   backbone: 0.00M params, lr=2e-05
train_fever |   lora: 7.11M params, lr=0.0005
train_fever | Using fused AdamW



  FEVER Training: mode=neural
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 7.11M / 442.18M (1.6%)
  epochs=5, bs=32x2=64, lr=2e-05, lr_lora=0.0005
  evidence_mode=gold, precision=bf16, compile=False
  total_steps=235, warmup=23

  Epoch 1/5: loss=1.0063 constraint=0.0000
  Step 50: loss=0.7206 | dev_acc=0.5840 ECE=0.0952
  Epoch 2/5: loss=0.6736 constraint=0.0000
  Step 100: loss=0.4745 | dev_acc=0.7880 ECE=0.0445
  Epoch 3/5: loss=0.4316 constraint=0.0000
  Step 150: loss=0.2313 | dev_acc=0.7680 ECE=0.1320
  Epoch 4/5: loss=0.3019 constraint=0.0000
  Step 200: loss=0.2004 | dev_acc=0.7700 ECE=0.1312
  Epoch 5/5: loss=0.2365 constraint=0.0000 | dev_acc=0.7667 ECE=0.1333

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 1.5884

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────
  Label Accuracy (GOLD evidence):

train_fever | Report saved to outputs_neural_3k/report.json


  Label Accuracy (GOLD evidence): 0.8400
  ECE: 0.0984
  Brier: 0.2560
    SUPPORTS: acc=0.8537 (n=41)
    REFUTES: acc=0.8148 (n=27)
    NOT ENOUGH INFO: acc=0.8438 (n=32)

  Training complete in 326.3s
  Best dev accuracy: 0.7880
  Output: outputs_neural_3k

  NEURAL BASELINE 3K RESULTS (6.0 min)
  Dev acc    : 0.7667
  Dev ECE    : 0.133265
  Best dev   : 0.788
    SUPPORTS            : 0.8970 (n=301)
    REFUTES             : 0.6558 (n=276)
    NOT ENOUGH INFO     : 0.7399 (n=323)

  Saved to results_neural_3k.json


In [ ]:
# ============================================================
# Cell 7: 3K NST-VERI (Constraint-Enhanced Training — THE CRITICAL RUN)
# ============================================================
# DeBERTa-v3-large + LoRA + verification heads + contrastive + adaptive lambda
# 3-phase: NLI+aux → +contrastive → +constraints
#
# WATCH FOR:
#   - constraint_loss > 0 (constraints must be active)
#   - fire_rate > 0 (constraints must fire)
#   - mean_lambda > 0.1 (constraints must have weight)
#   - dev_acc >= neural baseline (constraints must not hurt)
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI: Verification-Enhanced Constraint Training")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_3k.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_3k.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# ── Constraint activity analysis ──
train_log = results_veri_3k.get("train_log", [])
if train_log:
    phase3_entries = [e for e in train_log if e.get("phase", 0) >= 3]
    if phase3_entries:
        cst_losses = [e.get("loss_constraint", 0) for e in phase3_entries]
        lambdas = [e.get("mean_lambda", 0) for e in phase3_entries]
        print(f"\n  CONSTRAINT DIAGNOSTICS:")
        print(f"    Phase 3 entries   : {len(phase3_entries)}")
        print(f"    Constraint loss   : min={min(cst_losses):.4f} max={max(cst_losses):.4f} mean={sum(cst_losses)/len(cst_losses):.4f}")
        print(f"    Mean lambda       : min={min(lambdas):.4f} max={max(lambdas):.4f} mean={sum(lambdas)/len(lambdas):.4f}")
        if max(cst_losses) > 0.001:
            print(f"    CONSTRAINTS ARE ACTIVE")
        else:
            print(f"    WARNING: CONSTRAINTS STILL INACTIVE")
    else:
        print(f"\n  WARNING: No Phase 3 entries in training log")

# Constraint calibration
calib = results_veri_3k.get("constraint_calibration", {})
if calib:
    print(f"\n  CONSTRAINT CALIBRATION (on dev):")
    for cname, cstats in calib.items():
        print(f"    {cname}: precision={cstats.get('precision', 0):.3f} fire_rate={cstats.get('fire_rate', 0):.3f}")

with open("results_veri_3k.json", "w") as f:
    json.dump(results_veri_3k, f, indent=2, default=str)
print(f"\n  Saved to results_veri_3k.json")

train_fever_veri | TF32 enabled
train_fever_veri | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  3K NST-VERI: Verification-Enhanced Constraint Training


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (1586 with evidence text, 1414 without)
fever_dataset |   dev: 1000 examples (428 with evidence text, 572 without)
fever_dataset |   Split labelled_dev into dev (900) + dev_test (100)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: f36abc6968b9527d
fever_dataset |   dev_test hash: ae1018906c1b63d9
train_fever_veri | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 3000 examples
    With gold evidence: 1586 (52.9%)
    Label distribution:
      SUPPORTS               1594  (53.1%)
      REFUTES                 641  (21.4%)
      NOT ENOUGH INFO         765  (25.5%)
    Split hash: 971dd9bf22223f7f

  dev: 900 examples
    With gold evidence: 381 (42.3%)
    Label distribution:
      SUPPORTS                301  (33.4%)
      REFUTES                 276  (30.7%)
      NOT ENOUGH INFO         323  (35.9%)
    Split hash: f36abc6968b9527d

  dev_test: 100 examples
    With gold evidence: 47 (47.0%)
    Label distribution:
      SUPPORTS                 41  (41.0%)
      REFUTES                  27  (27.0%)
      NOT ENOUGH INFO          32  (32.0%)
    Split hash: ae1018906c1b63d9

  Evidence quality: 1572/3000 (52%) have >30 char evidence
    [0] SUPPORTS: Chitty Chitty Bang Bang is loosely based on another novel....
         Evidence: 
    [1] SUPPORTS: The Republic of Zambia is bordered to the west by Angola..

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever_veri | Class weights: [0.6273525953292847, 1.5600624084472656, 1.3071895837783813]
train_fever_veri |   backbone: 0.00M params, lr=2e-05
train_fever_veri |   lora: 7.11M params, lr=0.0005
train_fever_veri |   heads: 2.43M params, lr=0.001
train_fever_veri |   adaptive_lambda: 0.00M params, lr=0.003



  Constraint precision pre-check (2000 samples):
    NumericalConstraint            precision=0.726 fire_rate=0.057 [OK]
    NegationConstraint             precision=0.579 fire_rate=0.082 [OK]
    EntityOverlapConstraint        precision=0.416 fire_rate=0.353 [NOISY]
    EvidenceSufficiencyConstraint  precision=0.529 fire_rate=0.470 [OK]
    TemporalConstraint             precision=0.769 fire_rate=0.045 [OK]
    HedgeModalityConstraint        precision=0.500 fire_rate=0.002 [OK]

  NST-VERI Training: Verification-Enhanced FEVER
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 9.55M / 444.61M
  Adaptive lambda: 4551 params, lambda_max=0.8
  epochs=8, bs=32x2=64
  lr=2e-05, lr_lora=0.0005, lr_heads=0.001
  precision=bf16, compile=False
  Focal loss: no
  Phases: 1→NLI+aux, 2→+contrastive, 3→+constraints
  total_steps=376, warmup=37

  Phase 1 | Epoch 1/8 | β=1.00 γ=0.00 sched=0.00
  Epoch 1/8 (phase 1): loss=1.5166 nli=0.9432 aux=0.5735 con=0.0000 cst=0.0000 | dev_acc=0.6333

In [ ]:
# ============================================================
# Cell 8: 3K COMPARISON — Neural vs NST-VERI
# ============================================================
import json, os

experiments = {}
for name, path in [("neural_3k", "results_neural_3k.json"),
                   ("veri_3k", "results_veri_3k.json")]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  3K COMPARISON: Neural vs NST-VERI")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8}")
print(f"  {'─'*55}")

for name, r in experiments.items():
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "N/A")
    best = r.get("best_dev_acc", "N/A")
    ece = dev.get("ece", "N/A")
    acc_s = f"{acc:.4f}" if isinstance(acc, (int, float)) else str(acc)
    best_s = f"{best:.4f}" if isinstance(best, (int, float)) else str(best)
    ece_s = f"{ece:.4f}" if isinstance(ece, (int, float)) else str(ece)
    print(f"  {name:<25} {acc_s:>10} {best_s:>10} {ece_s:>8}")

# ── Signal assessment ──
if "neural_3k" in experiments and "veri_3k" in experiments:
    n_acc = experiments["neural_3k"].get("best_dev_acc", 0)
    v_acc = experiments["veri_3k"].get("best_dev_acc", 0)
    delta = v_acc - n_acc
    print(f"\n  Delta (VERI - Neural): {delta:+.4f}")
    if delta > 0.01:
        print(f"  SIGNAL: NST-VERI shows improvement. Full run justified.")
    elif delta > -0.01:
        print(f"  NEUTRAL: No clear signal yet. May need config tuning.")
    else:
        print(f"  WARNING: NST-VERI underperforms. Investigate before full run.")

  3K COMPARISON: Neural vs NST-VERI
  Method                       Dev Acc   Best Acc      ECE
  ───────────────────────────────────────────────────────
  neural_3k                     0.7667     0.7880   0.1333
  veri_3k                       0.7567     0.7820   0.1796

  Delta (VERI - Neural): -0.0060
  NEUTRAL: No clear signal yet. May need config tuning.


In [ ]:
# ============================================================
# DIAGNOSTIC: Why do SUPPORTS/REFUTES examples lack evidence?
# ============================================================
import json, sys, os
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_dataset import load_fever_splits, LABEL2ID
splits = load_fever_splits(max_train=3000, max_dev=1000, dev_test_ratio=0.1, seed=42)

# Analyze per-label evidence quality
for split_name in ["train", "dev"]:
    items = splits[split_name]
    print(f"\n{split_name} ({len(items)} examples):")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        label_items = [it for it in items if it["label"] == label]
        has_ev = [it for it in label_items if len(it.get("gold_evidence_text", "")) > 30]
        pct = 100 * len(has_ev) / max(1, len(label_items))
        print(f"  {label:<20}: {len(has_ev)}/{len(label_items)} ({pct:.0f}%) have evidence")
    
    # Show examples of SUPPORTS with no evidence
    no_ev_sup = [it for it in items if it["label"] == "SUPPORTS" and len(it.get("gold_evidence_text", "")) <= 30]
    if no_ev_sup:
        print(f"\n  Examples of SUPPORTS with NO evidence:")
        for it in no_ev_sup[:3]:
            print(f"    claim: {it['claim'][:80]}")
            print(f"    evidence: '{it['gold_evidence_text'][:100]}'")
            print(f"    id: {it['id']}")


fever_dataset | Loading FEVER from HuggingFace datasets...
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (1586 with evidence text, 1414 without)
fever_dataset |   dev: 1000 examples (428 with evidence text, 572 without)
fever_dataset |   Split labelled_dev into dev (900) + dev_test (100)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: f36abc6968b9527d
fever_dataset |   dev_test hash: ae1018906c1b63d9



train (3000 examples):
  SUPPORTS            : 1149/1594 (72%) have evidence
  REFUTES             : 423/641 (66%) have evidence
  NOT ENOUGH INFO     : 0/765 (0%) have evidence

  Examples of SUPPORTS with NO evidence:
    claim: Chitty Chitty Bang Bang is loosely based on another novel.
    evidence: ''
    id: 169162
    claim: The Republic of Zambia is bordered to the west by Angola.
    evidence: ''
    id: 191677
    claim: Miss Selfridge is located within the United Kingdom.
    evidence: ''
    id: 124198

dev (900 examples):
  SUPPORTS            : 216/301 (72%) have evidence
  REFUTES             : 162/276 (59%) have evidence
  NOT ENOUGH INFO     : 0/323 (0%) have evidence

  Examples of SUPPORTS with NO evidence:
    claim: Sancho Panza is a character in a novel written by an author.
    evidence: ''
    id: 173505
    claim: Taran Killam is an American writer.
    evidence: ''
    id: 39437
    claim: Kuching is the most populous city in Sarawak.
    evidence: ''
    id: 

In [ ]:
# ============================================================
# DIAGNOSTIC: Check HF dataset format and evidence fields
# ============================================================
from datasets import load_dataset
ds = load_dataset("fever", "v1.0", trust_remote_code=True)
print("Train columns:", ds["train"].column_names)
print("\nFirst 3 train rows:")
for i in range(3):
    row = ds["train"][i]
    print(f"  {i}: ", {k: str(v)[:80] for k, v in row.items()})

# Check specific IDs
from data.fever_wiki_cache import WikiCache
cache = WikiCache("/content/nst/data/fever_wiki.db")

bad_ids = {169162, 191677, 124198}
found = 0
for row in ds["train"]:
    if row["id"] in bad_ids:
        found += 1
        print(f"\nID={row['id']}: {row['claim'][:60]}")
        print(f"  All keys: {list(row.keys())}")
        print(f"  Full row: {row}")
        # Check wiki cache for the evidence_wiki_url
        wiki_url = row.get("evidence_wiki_url", "")
        sent_id = row.get("evidence_sentence_id", -1)
        if wiki_url:
            in_cache = wiki_url in cache
            looked_up = cache.lookup(wiki_url)
            n_sents = len(looked_up) if looked_up else 0
            print(f"  wiki_url='{wiki_url}' sent_id={sent_id} in_cache={in_cache} n_sents={n_sents}")
            if looked_up and isinstance(sent_id, int) and 0 <= sent_id < n_sents:
                print(f"    => '{looked_up[sent_id][:120]}'")
        if found >= 3:
            break


Train columns: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id']

First 3 train rows:
  0:  {'id': '75397', 'label': 'SUPPORTS', 'claim': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.', 'evidence_annotation_id': '92206', 'evidence_id': '104971', 'evidence_wiki_url': 'Nikolaj_Coster-Waldau', 'evidence_sentence_id': '7'}
  1:  {'id': '75397', 'label': 'SUPPORTS', 'claim': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.', 'evidence_annotation_id': '92206', 'evidence_id': '104971', 'evidence_wiki_url': 'Fox_Broadcasting_Company', 'evidence_sentence_id': '-1'}
  2:  {'id': '150448', 'label': 'SUPPORTS', 'claim': 'Roman Atwood is a content creator.', 'evidence_annotation_id': '174271', 'evidence_id': '187498', 'evidence_wiki_url': 'Roman_Atwood', 'evidence_sentence_id': '1'}

ID=169162: Chitty Chitty Bang Bang is loosely based on another novel.
  All keys: ['id', 'label', 'claim', 'evidence_annot

In [ ]:
# ============================================================
# Cell 11: PULL EVIDENCE FIX v2 (sent_id>=0 filter + sent_idx=-1 fallback)
# ============================================================
import subprocess, importlib, gc, sys

subprocess.run(["git", "pull", "--ff-only"], cwd=PROJ_ROOT, check=True)

# Force reload ALL project modules so fix takes effect
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

# Reload and rebuild splits
from data.fever_dataset import load_fever_splits
splits = load_fever_splits(max_train=3000, max_dev=1000)

for split_name in ["train", "dev"]:
    items = splits[split_name]
    n_with_evidence = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
    pct = n_with_evidence / len(items) * 100
    print(f"  {split_name}: {n_with_evidence}/{len(items)} ({pct:.1f}%) have evidence")

# Per-label breakdown
for split_name in ["train", "dev"]:
    print(f"\n{split_name} per-label:")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        items = [it for it in splits[split_name] if it["label"] == label]
        has_ev = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
        print(f"  {label}: {has_ev}/{len(items)} ({has_ev/len(items)*100:.1f}%)")
print("\nEvidence fix v2 applied ✓")

fever_dataset | Loading FEVER from HuggingFace datasets...
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (2235 with evidence text, 765 without)
fever_dataset |   dev: 1000 examples (645 with evidence text, 355 without)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: 339d321ff17fccd2


  train: 2235/3000 (74.5%) have evidence
  dev: 645/1000 (64.5%) have evidence

train per-label:
  SUPPORTS: 1594/1594 (100.0%)
  REFUTES: 641/641 (100.0%)
  NOT ENOUGH INFO: 0/765 (0.0%)

dev per-label:
  SUPPORTS: 342/342 (100.0%)
  REFUTES: 303/303 (100.0%)
  NOT ENOUGH INFO: 0/355 (0.0%)

Evidence fix v2 applied ✓


In [ ]:
# Quick diagnostic: check if the specific IDs we diagnosed now have evidence
test_ids = [169162, 191677, 124198]
for it in splits["train"]:
    if it["id"] in test_ids:
        ev = it.get("gold_evidence_text", "")
        print(f"ID {it['id']} ({it['label']}): evidence={ev[:80]!r}...")

# Also check: how many still have no evidence, broken by label?
from collections import Counter
for split_name in ["train", "dev"]:
    print(f"\n{split_name}:")
    for label in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        items = [it for it in splits[split_name] if it["label"] == label]
        has_ev = sum(1 for it in items if it.get("gold_evidence_text", "").strip())
        print(f"  {label}: {has_ev}/{len(items)} ({has_ev/len(items)*100:.1f}%) have evidence")

ID 169162 (SUPPORTS): evidence=''...
ID 191677 (SUPPORTS): evidence=''...
ID 124198 (SUPPORTS): evidence=''...

train:
  SUPPORTS: 1158/1594 (72.6%) have evidence
  REFUTES: 428/641 (66.8%) have evidence
  NOT ENOUGH INFO: 0/765 (0.0%) have evidence

dev:
  SUPPORTS: 246/342 (71.9%) have evidence
  REFUTES: 182/303 (60.1%) have evidence
  NOT ENOUGH INFO: 0/355 (0.0%) have evidence


In [ ]:
# Check which code is actually loaded and inspect the fix
import inspect, data.fever_dataset as fd
src = inspect.getsource(fd)
# Find the evidence resolution line
for i, line in enumerate(src.split('\n')):
    if 'effective_idx' in line or 'sent_idx >= 0' in line or '0 <= sent_idx' in line:
        print(f"Line {i}: {line}")

Line 94:                     effective_idx = sent_idx if isinstance(sent_idx, int) and sent_idx >= 0 else 0
Line 95:                     if effective_idx < len(sents):
Line 96:                         text = sents[effective_idx].strip()
Line 318:                         effective_idx = sent_idx if isinstance(sent_idx, int) and sent_idx >= 0 else 0
Line 319:                         if effective_idx < len(sents):
Line 320:                             text = sents[effective_idx].strip()


In [ ]:
# Deep debug: trace evidence resolution for ID 169162
target_id = 169162
rows = [r for r in ds["train"] if r["id"] == target_id]
print(f"Rows for {target_id}: {len(rows)}")
for r in rows[:3]:
    url = r["evidence_wiki_url"]
    sid = r["evidence_sentence_id"]
    print(f"  url={url!r} sent_id={sid}")
    sents = cache.lookup(url) if url else None
    print(f"  in_cache={sents is not None}, n_sents={len(sents) if sents else 0}")
    if sents:
        print(f"  sentence[0] = {sents[0][:100]!r}")

# Now check: does the code path actually reach our fix?
# Look at what format load_fever_splits uses
print(f"\nColumns: {ds['train'].column_names}")

# The key question: does the code use "flat" or "nested" format?
# Check if evidence_wiki_url is a string or list
r0 = ds["train"][0]
print(f"evidence_wiki_url type: {type(r0['evidence_wiki_url'])}")
print(f"evidence_sentence_id type: {type(r0['evidence_sentence_id'])}")

Rows for 169162: 2
  url='Chitty_Chitty_Bang_Bang' sent_id=-1
  in_cache=True, n_sents=9
  sentence[0] = 'Chitty Chitty Bang Bang is a 1968 British musical adventure fantasy film directed by Ken Hughes and '
  url='Chitty_Chitty_Bang_Bang' sent_id=-1
  in_cache=True, n_sents=9
  sentence[0] = 'Chitty Chitty Bang Bang is a 1968 British musical adventure fantasy film directed by Ken Hughes and '

Columns: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id']
evidence_wiki_url type: <class 'str'>
evidence_sentence_id type: <class 'int'>


In [ ]:
# ============================================================
# Cell 12: RE-RUN NEURAL BASELINE 3K (with fixed evidence)
# ============================================================
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NEURAL BASELINE v2: DeBERTa-v3-large + LoRA (fixed evidence)")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_3k_v2 = train_fever_nst(
    "configs/fever_neural_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_3k_v2.get("dev", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE 3K v2 RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_neural_3k_v2.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_3k_v2.json", "w") as f:
    json.dump(results_neural_3k_v2, f, indent=2, default=str)
print(f"\n  Saved to results_neural_3k_v2.json")

train_fever | TF32 enabled for matmul and cuDNN
train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  3K NEURAL BASELINE v2: DeBERTa-v3-large + LoRA (fixed evidence)


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (2235 with evidence text, 765 without)
fever_dataset |   dev: 1000 examples (645 with evidence text, 355 without)
fever_dataset |   Split labelled_dev into dev (900) + dev_test (100)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: f36abc6968b9527d
fever_dataset |   dev_test hash: ae1018906c1b63d9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 3000 examples
    With gold evidence: 2235 (74.5%)
    Label distribution:
      SUPPORTS               1594  (53.1%)
      REFUTES                 641  (21.4%)
      NOT ENOUGH INFO         765  (25.5%)
    Split hash: 971dd9bf22223f7f

  dev: 900 examples
    With gold evidence: 577 (64.1%)
    Label distribution:
      SUPPORTS                301  (33.4%)
      REFUTES                 276  (30.7%)
      NOT ENOUGH INFO         323  (35.9%)
    Split hash: f36abc6968b9527d

  dev_test: 100 examples
    With gold evidence: 68 (68.0%)
    Label distribution:
      SUPPORTS                 41  (41.0%)
      REFUTES                  27  (27.0%)
      NOT ENOUGH INFO          32  (32.0%)
    Split hash: ae1018906c1b63d9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever | Class weights: [0.6273525953292847, 1.5600624084472656, 1.3071895837783813]
train_fever | Periodic eval uses dev subset: 500/900 examples
train_fever | Params: 7.11M trainable / 442.18M total
train_fever |   backbone: 0.00M params, lr=2e-05
train_fever |   lora: 7.11M params, lr=0.0005
train_fever | Using fused AdamW



  FEVER Training: mode=neural
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 7.11M / 442.18M (1.6%)
  epochs=5, bs=32x2=64, lr=2e-05, lr_lora=0.0005
  evidence_mode=gold, precision=bf16, compile=False
  total_steps=235, warmup=23

  Epoch 1/5: loss=0.6718 constraint=0.0000
  Step 50: loss=0.2103 | dev_acc=0.9660 ECE=0.0158
  Epoch 2/5: loss=0.2095 constraint=0.0000
  Step 100: loss=0.2212 | dev_acc=0.9740 ECE=0.0101
  Epoch 3/5: loss=0.1389 constraint=0.0000
  Step 150: loss=0.1056 | dev_acc=0.9580 ECE=0.0193
  Epoch 4/5: loss=0.1225 constraint=0.0000
  Step 200: loss=0.1001 | dev_acc=0.9660 ECE=0.0146
  Epoch 5/5: loss=0.1091 constraint=0.0000 | dev_acc=0.9656 ECE=0.0179

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 1.2128

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────
  Label Accuracy (GOLD evidence):

train_fever | Report saved to outputs_neural_3k/report.json


  Label Accuracy (GOLD evidence): 0.9800
  ECE: 0.0073
  Brier: 0.0374
    SUPPORTS: acc=0.9756 (n=41)
    REFUTES: acc=0.9630 (n=27)
    NOT ENOUGH INFO: acc=1.0000 (n=32)

  Training complete in 380.8s
  Best dev accuracy: 0.9740
  Output: outputs_neural_3k

  NEURAL BASELINE 3K v2 RESULTS (7.0 min)
  Dev acc    : 0.9656
  Best dev   : 0.974
    SUPPORTS            : 0.9668 (n=301)
    REFUTES             : 0.9239 (n=276)
    NOT ENOUGH INFO     : 1.0000 (n=323)

  Saved to results_neural_3k_v2.json


In [ ]:
# ============================================================
# Cell 13: RE-RUN NST-VERI 3K (with fixed evidence)
# ============================================================
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  3K NST-VERI v2: DeBERTa-v3-large + LoRA + Constraints (fixed evidence)")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_3k_v2 = train_fever_veri(
    "configs/fever_veri_3k_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_3k_v2.get("dev", {})
print(f"\n{'='*65}")
print(f"  NST-VERI 3K v2 RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_veri_3k_v2.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Compare with neural
n_best = results_neural_3k_v2.get("best_dev_acc", 0)
v_best = results_veri_3k_v2.get("best_dev_acc", 0)
delta = v_best - n_best
print(f"\nDelta (VERI - Neural): {delta:+.4f}")
if v_best >= 0.90:
    print("*** TARGET REACHED: 90%+ accuracy! ***")

with open("results_veri_3k_v2.json", "w") as f:
    json.dump(results_veri_3k_v2, f, indent=2, default=str)
print(f"  Saved to results_veri_3k_v2.json")

train_fever_veri | TF32 enabled
train_fever_veri | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  3K NST-VERI v2: DeBERTa-v3-large + LoRA + Constraints (fixed evidence)


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 3000 examples (2235 with evidence text, 765 without)
fever_dataset |   dev: 1000 examples (645 with evidence text, 355 without)
fever_dataset |   Split labelled_dev into dev (900) + dev_test (100)
fever_dataset |   train hash: 971dd9bf22223f7f
fever_dataset |   dev hash: f36abc6968b9527d
fever_dataset |   dev_test hash: ae1018906c1b63d9
train_fever_veri | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 3000 examples
    With gold evidence: 2235 (74.5%)
    Label distribution:
      SUPPORTS               1594  (53.1%)
      REFUTES                 641  (21.4%)
      NOT ENOUGH INFO         765  (25.5%)
    Split hash: 971dd9bf22223f7f

  dev: 900 examples
    With gold evidence: 577 (64.1%)
    Label distribution:
      SUPPORTS                301  (33.4%)
      REFUTES                 276  (30.7%)
      NOT ENOUGH INFO         323  (35.9%)
    Split hash: f36abc6968b9527d

  dev_test: 100 examples
    With gold evidence: 68 (68.0%)
    Label distribution:
      SUPPORTS                 41  (41.0%)
      REFUTES                  27  (27.0%)
      NOT ENOUGH INFO          32  (32.0%)
    Split hash: ae1018906c1b63d9

  Evidence quality: 2211/3000 (74%) have >30 char evidence
    [0] SUPPORTS: Chitty Chitty Bang Bang is loosely based on another novel....
         Evidence: Chitty Chitty Bang Bang is a 1968 British musical adventure fantasy film dire

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever_veri | Class weights: [0.6273525953292847, 1.5600624084472656, 1.3071895837783813]
train_fever_veri |   backbone: 0.00M params, lr=2e-05
train_fever_veri |   lora: 7.11M params, lr=0.0005
train_fever_veri |   heads: 2.43M params, lr=0.001
train_fever_veri |   adaptive_lambda: 0.00M params, lr=0.003



  Constraint precision pre-check (2000 samples):
    NumericalConstraint            precision=0.685 fire_rate=0.083 [OK]
    NegationConstraint             precision=0.556 fire_rate=0.086 [OK]
    EntityOverlapConstraint        precision=0.703 fire_rate=0.209 [OK]
    EvidenceSufficiencyConstraint  precision=0.967 fire_rate=0.258 [OK]
    TemporalConstraint             precision=0.792 fire_rate=0.086 [OK]
    HedgeModalityConstraint        precision=0.375 fire_rate=0.004 [NOISY]

  NST-VERI Training: Verification-Enhanced FEVER
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 9.55M / 444.61M
  Adaptive lambda: 4551 params, lambda_max=0.8
  epochs=8, bs=32x2=64
  lr=2e-05, lr_lora=0.0005, lr_heads=0.001
  precision=bf16, compile=False
  Focal loss: no
  Phases: 1→NLI+aux, 2→+contrastive, 3→+constraints
  total_steps=376, warmup=37

  Phase 1 | Epoch 1/8 | β=1.00 γ=0.00 sched=0.00
  Epoch 1/8 (phase 1): loss=1.3296 nli=0.7281 aux=0.6015 con=0.0000 cst=0.0000 | dev_acc=0.9511

In [ ]:
# Quick results summary
import json
with open("results_veri_3k_v2.json") as f:
    r = json.load(f)
dev = r.get("dev", {})
print(f"NST-VERI 3K v2:")
print(f"  Best dev acc: {r.get('best_dev_acc', 'N/A')}")
print(f"  Final dev acc: {dev.get('accuracy', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

n_best = results_neural_3k_v2.get("best_dev_acc", 0)
v_best = r.get("best_dev_acc", 0)
print(f"\nNeural best: {n_best:.4f}")
print(f"VERI best:   {v_best:.4f}")
print(f"Delta:       {v_best - n_best:+.4f}")
if v_best >= 0.90:
    print("*** TARGET REACHED: 90%+ ***")

NST-VERI 3K v2:
  Best dev acc: 0.974
  Final dev acc: 0.9589
    SUPPORTS: 0.9568 (n=301)
    REFUTES: 0.9130 (n=276)
    NOT ENOUGH INFO: 1.0000 (n=323)

Neural best: 0.9740
VERI best:   0.9740
Delta:       +0.0000
*** TARGET REACHED: 90%+ ***


In [ ]:
# ============================================================
# Cell 9: FULL NST-VERI RUN (Only after 3k shows signal)
# ============================================================
# Run ONLY after Cell 8 comparison shows VERI >= Neural on 3k.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NST-VERI: DeBERTa-v3-large + LoRA + Verification")
print("=" * 65)

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri_full = train_fever_veri(
    "configs/fever_gold_nst_veri_a100.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_veri_full.get("dev", {})
print(f"\n{'='*65}")
print(f"  FULL NST-VERI RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
print(f"  Best dev   : {results_veri_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Constraint diagnostics
train_log = results_veri_full.get("train_log", [])
phase3 = [e for e in train_log if e.get("phase", 0) >= 3]
if phase3:
    cst = [e.get("loss_constraint", 0) for e in phase3]
    lam = [e.get("mean_lambda", 0) for e in phase3]
    print(f"\n  Constraint loss: {min(cst):.4f}—{max(cst):.4f} (mean {sum(cst)/len(cst):.4f})")
    print(f"  Mean lambda:     {min(lam):.4f}—{max(lam):.4f} (mean {sum(lam)/len(lam):.4f})")

# Held-out dev_test
dt = results_veri_full.get("dev_test")
if dt:
    print(f"\n  Held-out dev_test accuracy: {dt.get('accuracy', '?')}")

with open("results_veri_full.json", "w") as f:
    json.dump(results_veri_full, f, indent=2, default=str)
print(f"\n  Saved to results_veri_full.json")

train_fever_veri | TF32 enabled
train_fever_veri | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  FULL NST-VERI: DeBERTa-v3-large + LoRA + Verification


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (109810 with evidence text, 35639 without)
fever_dataset |   dev: 19998 examples (13332 with evidence text, 6666 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever_veri | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 109810 (75.5%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 11983 (66.6%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 1349 (67.5%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9

  Evidence quality: 109214/145449 (75%) have >30 char evidence
    [0] SUPPORTS: Chris Hemsworth appeared in A Perfect Getaway....
         Evidence: Hemsworth has also appeared in the science fiction action film Star Trek -

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Gradient checkpointing enabled
fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever_veri | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever_veri |   backbone: 0.00M params, lr=1e-05
train_fever_veri |   lora: 7.11M params, lr=0.0003
train_fever_veri |   heads: 2.43M params, lr=0.0005
train_fever_veri |   adaptive_lambda: 0.00M params, lr=0.001



  Constraint precision pre-check (2000 samples):
    NumericalConstraint            precision=0.805 fire_rate=0.110 [OK]
    NegationConstraint             precision=0.423 fire_rate=0.061 [NOISY]
    EntityOverlapConstraint        precision=0.583 fire_rate=0.138 [OK]
    EvidenceSufficiencyConstraint  precision=0.978 fire_rate=0.158 [OK]
    TemporalConstraint             precision=0.872 fire_rate=0.094 [OK]
    HedgeModalityConstraint        precision=0.125 fire_rate=0.004 [NOISY]

  NST-VERI Training: Verification-Enhanced FEVER
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 9.55M / 444.61M
  Adaptive lambda: 4551 params, lambda_max=1.5
  epochs=5, bs=32x2=64
  lr=1e-05, lr_lora=0.0003, lr_heads=0.0005
  precision=bf16, compile=False
  Focal loss: no
  Phases: 1→NLI+aux, 2→+contrastive, 3→+constraints
  total_steps=11365, warmup=681

  Phase 1 | Epoch 1/5 | β=1.00 γ=0.00 sched=0.00
    Step 250: loss=0.8502 nli=0.2711 aux=0.5791 con=0.0000 cst=0.0000 | dev_acc=0.9690 E

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 10: FULL NEURAL BASELINE (Fair Comparison)
# ============================================================
# Same DeBERTa-v3-large + LoRA, NO constraints.
import time, json, gc, sys

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

print("=" * 65)
print("  FULL NEURAL BASELINE: DeBERTa-v3-large + LoRA")
print("=" * 65)

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural_full = train_fever_nst(
    "configs/fever_gold_neural.yaml",
    config_overrides=GPU_OVERRIDES,
)
elapsed = time.time() - t0

dev = results_neural_full.get("dev", {})
print(f"\n{'='*65}")
print(f"  FULL NEURAL RESULTS ({elapsed/60:.1f} min)")
print(f"{'='*65}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Best dev   : {results_neural_full.get('best_dev_acc', 'N/A')}")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural_full.json", "w") as f:
    json.dump(results_neural_full, f, indent=2, default=str)
print(f"\n  Saved to results_neural_full.json")

In [ ]:
# ============================================================
# Cell 11: FINAL COMPARISON & HONEST REPORT
# ============================================================
import json, os, glob

experiments = {}
for name, path in [
    ("neural_3k", "results_neural_3k.json"),
    ("veri_3k", "results_veri_3k.json"),
    ("neural_full", "results_neural_full.json"),
    ("veri_full", "results_veri_full.json"),
]:
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)

print(f"{'='*65}")
print(f"  FINAL RESULTS — FEVER Gold Evidence Label Accuracy")
print(f"{'='*65}")
print(f"  {'Method':<25} {'Dev Acc':>10} {'Best Acc':>10} {'ECE':>8} {'Brier':>8}")
print(f"  {'─'*63}")

for name, r in sorted(experiments.items()):
    dev = r.get("dev", {})
    acc = dev.get("accuracy", "?")
    best = r.get("best_dev_acc", "?")
    ece = dev.get("ece", "?")
    brier = dev.get("brier", "?")
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
    print(f"  {name:<25} {fmt(acc):>10} {fmt(best):>10} {fmt(ece):>8} {fmt(brier):>8}")

# Per-label breakdown for full runs
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dev = experiments[name].get("dev", {})
        print(f"\n  {name} per-label:")
        for label, stats in dev.get("per_label", {}).items():
            print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Held-out dev_test comparison
for name in ["neural_full", "veri_full"]:
    if name in experiments:
        dt = experiments[name].get("dev_test")
        if dt:
            print(f"\n  {name} held-out dev_test: {dt.get('accuracy', '?')}")

print(f"\n{'='*65}")
print(f"  HONEST ASSESSMENT")
print(f"{'='*65}")
if "veri_full" in experiments:
    final_acc = experiments["veri_full"].get("best_dev_acc", 0)
    if final_acc >= 0.90:
        print(f"  TARGET REACHED: {final_acc:.4f} >= 0.90")
    else:
        print(f"  TARGET NOT YET REACHED: {final_acc:.4f} < 0.90")
        print(f"  Next steps: analyze per-label failures, tune constraints, retrain")

In [ ]:
# ============================================================
# Cell 12: SEED SWEEP (Reproducibility — 3 seeds)
# ============================================================
# Run after achieving 90%+ on seed=42 to verify result is real.
import time, json, gc, sys

seeds = [42, 43, 44]
seed_results = {}

for seed in seeds:
    for mod in list(sys.modules.keys()):
        if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                            "logic.", "symbolic.", "retrieval."]):
            del sys.modules[mod]
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print(f"  Seed {seed} — Full NST-VERI")
    print(f"{'='*50}")

    t0 = time.time()
    from training.train_fever_veri import train_fever_veri
    r = train_fever_veri(
        "configs/fever_gold_nst_veri_a100.yaml",
        config_overrides={
            **GPU_OVERRIDES,
            "seed": seed,
            "io": {"out_dir": f"outputs_veri_seed{seed}"},
        },
    )
    elapsed = time.time() - t0
    seed_results[seed] = r
    dev = r.get("dev", {})
    print(f"  Seed {seed}: acc={dev.get('accuracy','?')} best={r.get('best_dev_acc','?')} ({elapsed/60:.1f}min)")

# Summary
accs = [seed_results[s].get("best_dev_acc", 0) for s in seeds]
import numpy as np
print(f"\n{'='*50}")
print(f"  SEED SWEEP RESULTS")
print(f"  Accs: {accs}")
print(f"  Mean: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# Cell 13: SAVE ARTIFACTS — Download from Colab
# ============================================================
import json, os, shutil, glob

# Gather all result files
artifacts = glob.glob("results_*.json") + glob.glob("outputs_*/report.json")
print(f"Result artifacts: {artifacts}")

# Create a zip for easy download
artifact_dir = "nst_results"
os.makedirs(artifact_dir, exist_ok=True)
for f in artifacts:
    shutil.copy(f, artifact_dir)
# Add configs used
for cfg in glob.glob("configs/fever_*3k*.yaml") + glob.glob("configs/fever_gold_nst_veri*.yaml"):
    shutil.copy(cfg, artifact_dir)

shutil.make_archive("nst_results", "zip", ".", artifact_dir)
print(f"Download: nst_results.zip")

# Colab download helper
try:
    from google.colab import files
    files.download("nst_results.zip")
except ImportError:
    print("Not on Colab — download nst_results.zip manually")

In [ ]:
# ============================================================
# Cell 14: LEAKAGE AUDIT — Verify no data contamination
# ============================================================
# Run AFTER training to verify results are honest.
import json, sys, os

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_dataset import load_fever_splits

# Load same splits with same seed
splits = load_fever_splits(max_train=None, max_dev=None, dev_test_ratio=0.1, seed=42)

train_claims = {it["claim"] for it in splits["train"]}
dev_claims = {it["claim"] for it in splits["dev"]}
dev_test_claims = {it["claim"] for it in splits.get("dev_test", [])}

# Check overlaps
train_dev_overlap = train_claims & dev_claims
train_devtest_overlap = train_claims & dev_test_claims
dev_devtest_overlap = dev_claims & dev_test_claims

print(f"{'='*50}")
print(f"  LEAKAGE AUDIT")
print(f"{'='*50}")
print(f"  Train size     : {len(splits['train'])}")
print(f"  Dev size       : {len(splits['dev'])}")
print(f"  Dev-test size  : {len(splits.get('dev_test', []))}")
print(f"  Train∩Dev      : {len(train_dev_overlap)} overlapping claims")
print(f"  Train∩DevTest  : {len(train_devtest_overlap)} overlapping claims")
print(f"  Dev∩DevTest    : {len(dev_devtest_overlap)} overlapping claims")

if len(train_dev_overlap) == 0 and len(train_devtest_overlap) == 0:
    print(f"\n  LEAKAGE CHECK PASSED — No contamination detected")
else:
    print(f"\n  WARNING: Potential leakage detected!")
    if train_dev_overlap:
        print(f"    Example overlap: {list(train_dev_overlap)[:3]}")